# VERIMIND: Complete Project Walkthrough

This notebook is the guided entry point for **VERIMIND**, a local-first prototype that audits factual claims in LLM output. It demonstrates the maintained modules in `verimind/` and includes a complete source-code reference for every project Python file.

## Learning path

1. Setup and project map  -> 2. Claim segmentation  -> 3. Evidence retrieval  -> 4. NLI-style classification  -> 5. Risk scoring  -> 6. End-to-end verification  -> 7. Storage and API  -> 8. Training, evaluation, and tests

> **Run from top to bottom.** The normal walkthrough is offline-safe: web search is disabled unless configured.


## 1. Architecture at a glance

```text
Text / LLM answer
  -> split_claims()                 filters greetings, opinions, questions, and short fragments
  -> HybridRetriever.search()       ranks local TF-IDF evidence; optionally adds web evidence
  -> NLIModel.predict_proba()       entailment / contradiction / neutral probabilities
  -> RiskScorer.score_claim()       explainable 0-100 risk and verdict
  -> VerificationPipeline           document summary and claim-level audit trail
  -> FastAPI / SQLite / Streamlit    optional product interfaces
```

The maintained implementation is `verimind/`. Root-level `pipeline.py`, `judge.py`, and the older `debug_pipeline.ipynb` are legacy/debug utilities—not the modules used by the FastAPI application and tests.


In [ ]:
from pathlib import Path
import os
import sys

# This notebook belongs in the project root, but also works from another CWD.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "verimind").is_dir():
    candidates = [Path(r"D:/VeriMind/Code"), Path.cwd().resolve() / "Code"]
    PROJECT_ROOT = next((p for p in candidates if (p / "verimind").is_dir()), PROJECT_ROOT)
assert (PROJECT_ROOT / "verimind").is_dir(), "Open the notebook from the VERIMIND project folder."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])


### Dependencies

Use the project virtual environment as the Jupyter kernel. From `D:\VeriMind\Code`:

```powershell
.\venv\Scripts\python.exe -m pip install -r requirements.txt
.\venv\Scripts\python.exe -m pip install ipykernel
.\venv\Scripts\python.exe -m ipykernel install --user --name verimind --display-name "Python (VERIMIND)"
```

Select **Python (VERIMIND)** in Jupyter/VS Code. This notebook never installs packages automatically.


In [ ]:
import fastapi, numpy, pandas, sklearn
print("Core packages:")
for module in (fastapi, numpy, pandas, sklearn):
    print(f"- {module.__name__}: {getattr(module, '__version__', 'installed')}")

project_map = {
    "verimind/": "Core claim, retrieval, NLI, risk, config, and storage modules.",
    "api/": "FastAPI routes, schemas, API-key and rate-limit dependencies.",
    "dashboard/": "Streamlit browser interface that calls the API.",
    "scripts/": "CLI demo, train, evaluate, and benchmark entry points.",
    "data/": "Corpus, benchmark JSONL files, retrieval index, and SQLite database.",
    "artifacts/": "Persisted trained model artifact.",
    "tests/": "Pytest validation of core workflow and API.",
}
pandas.DataFrame(project_map.items(), columns=["Path", "Purpose"])


## 2. Configuration and safe local defaults

`.env.example` lists settings. Do **not** display or commit `.env`, because it can contain secrets. The default `SEARCH_BACKEND=null` means no external network call; local evidence retrieval still works. Use `SEARCH_BACKEND=tavily` plus `SEARCH_API_KEY` only when live web retrieval is intended.


In [ ]:
from verimind.config import Settings
settings = Settings.from_env()
{
    "search_backend": settings.search_backend,
    "log_level": settings.log_level,
    "max_claims_per_doc": settings.max_claims_per_doc,
    "max_evidence_per_claim": settings.max_evidence_per_claim,
    "risk_threshold": settings.risk_threshold,
    "api_key_configured": bool(settings.api_key),
    "search_key_configured": bool(settings.search_api_key),
}


## 3. Claim segmentation

`split_claims()` splits sentences and filters non-factual material: greetings, questions, opinion/uncertainty phrases, and statements shorter than three words. It is transparent but not a full linguistic claim extractor.


In [ ]:
from verimind.claim_segmentation import split_claims
sample_text = "Hello! Paris is the capital of France. The Eiffel Tower is in Paris. Is Berlin in Germany? I think France is wonderful. Thanks!"
claims = split_claims(sample_text)
print("Input:", sample_text)
print("\nClaims retained:")
for number, claim in enumerate(claims, 1):
    print(f"{number}. {claim}")


## 4. Evidence: local knowledge base and TF-IDF ranking

The small built-in `KnowledgeBase` is the offline corpus used by the demo pipeline. `LocalEvidenceRetriever` vectorizes all facts with TF-IDF and ranks them by cosine similarity. `HybridRetriever` combines those results with an optional web backend and deduplicates source URLs.


In [ ]:
from verimind.knowledge_base import KnowledgeBase
from verimind.retrieval import HybridRetriever, LocalEvidenceRetriever
kb = KnowledgeBase()
pandas.DataFrame(kb.facts)


In [ ]:
query = "Paris is the capital of France."
retriever = LocalEvidenceRetriever(kb)
evidence = retriever.search(query, k=3)
pandas.DataFrame(evidence)[["text", "domain", "score", "source"]]


In [ ]:
hybrid = HybridRetriever(kb)
print("Queries generated for one claim:")
print(*hybrid.build_queries(query), sep="\n- ")
print("\nConfigured web backend:", type(hybrid.backend).__name__)
print("Hybrid evidence count:", len(hybrid.search(query, k=3)))


## 5. NLI-style classification

`NLIModel` is **not** a transformer. It learns from a small balanced synthetic data set with word/character TF-IDF features and an `MLPClassifier`. Its predictions are blended with lexical checks for exact matches, entity conflicts, numeric mismatches, and negation mismatches. This makes the baseline offline and explainable, but limited.


In [ ]:
from verimind.data import build_synthetic_dataset
from verimind.nli_model import NLIModel

dataset = build_synthetic_dataset()
pairs_frame = pandas.DataFrame(dataset["pairs"], columns=["claim", "evidence"])
pairs_frame["label"] = dataset["labels"]
print("Training examples by label:")
display(pairs_frame["label"].value_counts().rename_axis("label").reset_index(name="count"))
display(pairs_frame.head())

model = NLIModel().fit(dataset["pairs"], dataset["labels"])
print("Learned class order:", model.classes_)


In [ ]:
examples = [
    ("Paris is the capital of France.", "Paris is the capital city of France."),
    ("Paris is the capital of France.", "Paris is the capital of Germany."),
    ("Paris is the capital of France.", "The Eiffel Tower is in Paris, France."),
]
probabilities = model.predict_proba(examples)
classification = pandas.DataFrame(probabilities, columns=model.classes_)
classification.insert(0, "prediction", model.predict(examples))
classification.insert(0, "evidence", [item[1] for item in examples])
classification.insert(0, "claim", [item[0] for item in examples])
classification


## 6. Explainable risk scoring

Risk combines NLI probabilities and average retrieval quality. Exact support gets an additional boost; the result is bounded to 0-100. Verdict rules: **supported** means high support with low contradiction, **unsupported** means strong contradiction, and **uncertain** means weak or insufficient evidence.


In [ ]:
from verimind.risk_scoring import RiskScorer
scorer = RiskScorer()
scenarios = [
    ("supported", evidence[:1], {"entailment": 0.92, "contradiction": 0.03, "neutral": 0.05}),
    ("unsupported", evidence[:1], {"entailment": 0.05, "contradiction": 0.85, "neutral": 0.10}),
    ("uncertain", [], {"entailment": 0.15, "contradiction": 0.15, "neutral": 0.70}),
]
rows = []
for expected, scenario_evidence, probs in scenarios:
    rows.append({"illustrative_input": expected, **scorer.score_claim(query, scenario_evidence, probs)})
pandas.DataFrame(rows)[["illustrative_input", "verdict", "risk_score", "evidence_count", "why"]]


## 7. End-to-end verification pipeline

`VerificationPipeline` orchestrates all preceding steps. It trains the deterministic baseline from synthetic data when constructed. Each retained claim gets up to three evidence items, aggregated class probabilities, an explanation, and a risk score. The document score is the mean claim risk.


In [ ]:
from verimind.pipeline import VerificationPipeline
pipeline = VerificationPipeline()
document = "Paris is the capital of France. The Eiffel Tower is in Paris. The Eiffel Tower is in Berlin."
report = pipeline.verify_text(document)
print("Document summary:")
display(pandas.DataFrame([report["document_summary"]]))
claim_view = pandas.DataFrame([
    {
        "claim": item["claim"], "verdict": item["verdict"], "risk_score": item["risk_score"],
        "latency_ms": item["verification_latency_ms"], "evidence_count": len(item["evidence"]),
        **{f"p_{key}": round(value, 3) for key, value in item["nli_probs"].items()}, "why": item["why"],
    }
    for item in report["claims"]
])
claim_view


In [ ]:
# Inspect the evidence trail for every claim.
for item in report["claims"]:
    print(f"\nClaim: {item['claim']}\nVerdict: {item['verdict']} | Risk: {item['risk_score']}")
    display(pandas.DataFrame(item["evidence"])[["text", "domain", "score", "source"]])


In [ ]:
# Replace this string and run the cell to audit your own text.
your_text = "France is in Europe. Berlin is the capital of Germany. Paris is the capital of Germany."
your_report = pipeline.verify_text(your_text)
pandas.DataFrame([
    {"claim": row["claim"], "verdict": row["verdict"], "risk_score": row["risk_score"], "why": row["why"]}
    for row in your_report["claims"]
])


## 8. Optional persistence: SQLite report storage

The API saves reports to `data/verimind.db`. The following cell **writes** one new local record and reads it back. It is commented out to keep a learning-only run free of data changes.


In [ ]:
# OPTIONAL — this writes to data/verimind.db.
# from verimind.storage import get_report, save_report
# report_id = save_report(document, report)
# restored_report = get_report(report_id)
# print(f"Saved report ID: {report_id}")
# assert restored_report == report


## 9. Product interfaces: FastAPI and Streamlit

`api/main.py` exposes `GET /health`, `POST /verify`, `GET /report/{report_id}`, and `GET /metrics`. Authentication uses `x-api-key`; the local demo key is `demo-key`, but deployments should set a different `API_KEY`. The Streamlit dashboard submits text to the API.

Run these in separate terminals from the project root:

```powershell
.\venv\Scripts\python.exe -m uvicorn api.main:app --reload
.\venv\Scripts\python.exe -m streamlit run dashboard/app.py
```

Browse API docs at `http://localhost:8000/docs` and the dashboard at `http://localhost:8501`.


In [ ]:
# In-process API health check: no server is required and no report is written.
from fastapi.testclient import TestClient
from api.main import app
client = TestClient(app)
health = client.get("/health")
assert health.status_code == 200
health.json()


## 10. Training, evaluation, benchmarking, and tests

The notebook already trained a model in memory. The project also provides scripts:

```powershell
.\venv\Scripts\python.exe scripts\train.py       # overwrites artifacts/nli_model.pkl
.\venv\Scripts\python.exe scripts\evaluate.py    # deterministic representative holdout
.\venv\Scripts\python.exe scripts\benchmark.py   # baseline plus pipeline sample risks
.\venv\Scripts\python.exe -m pytest -q             # core/API regression checks
```

The training script writes `artifacts/nli_model.pkl`; run it only when you intentionally want to replace that artifact. The cell below reproduces the evaluation workflow without writing files.


In [ ]:
labels = dataset["labels"]
test_indices = {next(i for i, label in enumerate(labels) if label == target) for target in set(labels)}
train_pairs = [pair for i, pair in enumerate(dataset["pairs"]) if i not in test_indices]
train_labels = [label for i, label in enumerate(labels) if i not in test_indices]
test_pairs = [pair for i, pair in enumerate(dataset["pairs"]) if i in test_indices]
test_labels = [label for i, label in enumerate(labels) if i in test_indices]
holdout_model = NLIModel().fit(train_pairs, train_labels)
predictions = holdout_model.predict(test_pairs)
evaluation = pandas.DataFrame({"claim": [p[0] for p in test_pairs], "expected": test_labels, "predicted": predictions})
accuracy = (evaluation["expected"] == evaluation["predicted"]).mean()
print(f"Representative holdout accuracy: {accuracy:.1%}")
evaluation


## 11. Limitations and responsible interpretation

VERIMIND is a learning/prototype system, not a factual oracle.

- The offline knowledge base has only four illustrative facts, so unfamiliar claims often become **uncertain**.
- Its NLI classifier uses a small synthetic dataset; it is not a broadly evaluated large NLI model.
- Retrieval relevance is lexical TF-IDF; relevance is not proof or source authority.
- Claim segmentation is heuristic and can miss compound or implied claims.
- SQLite storage and in-memory rate limiting are local-development implementations.
- Optional Tavily results depend on an external service and still need human source review.

**Recommended next steps:** curate a larger corpus, use independently evaluated NLI data/models, assess source quality and citations, improve claim extraction, and evaluate with held-out real-world benchmarks.

---

### Where to go next

- Core orchestration: `verimind/pipeline.py`
- Retrieval: `verimind/retrieval.py` and `verimind/web_search.py`
- Model and safeguards: `verimind/nli_model.py`, `verimind/features.py`
- API: `api/main.py`
- Interface: `dashboard/app.py`
- Regression suite: `tests/test_verimind.py`


# Complete Python Source Reference\n\nThe following section embeds the full contents of **every project `.py` file** (excluding virtual environments, Git metadata, and cache folders) as it existed when this notebook was generated. Each file has its own code cell. These are reference cells: they have the `source-reference` and `do-not-run` tags because running every module, test, and CLI entry point sequentially in a notebook is not a safe execution workflow.\n\nEmbedded files: **31**.\n

## `api/deps.py`\n\nFull source code from this project file.\n

In [ ]:
"""API dependencies such as auth and rate limiting."""

from __future__ import annotations

from collections import defaultdict, deque
from time import time

from fastapi import Depends, HTTPException, Request


class InMemoryRateLimiter:
    """Simple token-bucket-like limiter keyed by API key."""

    def __init__(self, max_requests: int = 60, window_seconds: int = 60) -> None:
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests: dict[str, deque[float]] = defaultdict(deque)

    def check(self, api_key: str) -> None:
        now = time()
        bucket = self.requests[api_key]
        bucket.append(now)
        while bucket and now - bucket[0] > self.window_seconds:
            bucket.popleft()
        if len(bucket) > self.max_requests:
            raise HTTPException(status_code=429, detail="Rate limit exceeded")


rate_limiter = InMemoryRateLimiter()


def get_api_key(request: Request) -> str:
    """Validate the x-api-key header or allow the default local demo key."""
    configured = None
    try:
        import os

        configured = os.getenv("API_KEY")
    except Exception:
        configured = None

    api_key = request.headers.get("x-api-key")
    if api_key is None:
        if configured is None or configured == "demo-key":
            return "demo-key"
        raise HTTPException(status_code=401, detail="Missing API key")

    if configured is not None and configured != "demo-key" and api_key != configured:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return api_key


def require_api_key(request: Request) -> str:
    """Authenticate and enforce rate limits."""
    api_key = get_api_key(request)
    rate_limiter.check(api_key)
    return api_key


## `api/main.py`\n\nFull source code from this project file.\n

In [ ]:
"""FastAPI app exposing the VERIMIND verification endpoint."""

from __future__ import annotations

from typing import Any

from fastapi import Depends, FastAPI, HTTPException

from api.deps import require_api_key
from api.schemas import VerifyRequest, VerifyResponse
from verimind.pipeline import VerificationPipeline
from verimind.storage import get_report, save_report

app = FastAPI(title="VERIMIND API", version="0.1.0")

pipeline = VerificationPipeline()


@app.get("/health")
def health() -> dict[str, str]:
    """Return liveness/readiness status."""
    return {"status": "ok"}


@app.post("/verify", response_model=VerifyResponse)
def verify_document(payload: VerifyRequest, _: str = Depends(require_api_key)) -> dict[str, Any]:
    """Verify a document or answer and return claim-level evidence-backed results."""
    report = pipeline.verify_text(payload.text)
    report_id = save_report(payload.text, report)
    report["report_id"] = report_id
    return report


@app.get("/report/{report_id}")
def fetch_report(report_id: int, _: str = Depends(require_api_key)) -> dict[str, Any]:
    """Fetch a prior report by id."""
    report = get_report(report_id)
    if report is None:
        raise HTTPException(status_code=404, detail="Report not found")
    return report


@app.get("/metrics")
def metrics() -> dict[str, Any]:
    """Return simple operational metrics."""
    return {"requests_served": 1, "avg_latency_ms": 0.0, "flagged_ratio": 0.0}


@app.get("/docs")
def docs_route() -> dict[str, str]:
    """Compatibility route to the generated docs."""
    return {"docs_url": "/docs"}


## `api/schemas.py`\n\nFull source code from this project file.\n

In [ ]:
"""Pydantic schemas for the verification API."""

from __future__ import annotations

from pydantic import BaseModel, Field


class VerifyRequest(BaseModel):
    """Request body for verification."""

    text: str = Field(..., min_length=1, max_length=20000)


class VerifyResponse(BaseModel):
    """Response payload for verification."""

    document_summary: dict
    claims: list[dict]


## `dashboard/app.py`\n\nFull source code from this project file.\n

In [ ]:
"""Minimal Streamlit dashboard for interacting with VERIMIND."""

from __future__ import annotations

import json

import requests
import streamlit as st

API_URL = "http://localhost:8000/verify"

st.title("VERIMIND Dashboard")
text = st.text_area("LLM output to verify", height=250)
if st.button("Verify"):
    if not text.strip():
        st.warning("Please enter text to verify.")
    else:
        response = requests.post(API_URL, json={"text": text}, headers={"x-api-key": "demo-key"}, timeout=30)
        if response.status_code != 200:
            st.error(f"API error: {response.status_code} - {response.text}")
        else:
            payload = response.json()
            st.metric("Overall risk", payload["document_summary"]["overall_risk_score"])
            for index, claim in enumerate(payload["claims"], start=1):
                st.markdown(f"### Claim {index}: {claim['claim']}")
                st.write({
                    "verdict": claim["verdict"],
                    "risk_score": claim["risk_score"],
                    "why": claim["why"],
                })
                for evidence in claim["evidence"]:
                    st.write(f"- {evidence.get('text', '')} [{evidence.get('source', 'unknown')}]")


## Independent Wikipedia corpus and index

Build a reservoir sample from the FEVER Wikipedia dump with `python scripts/build_wikipedia_index.py --pages 5000 --seed 42`. This path uses the same recorded MiniLM embedding model, normalization, and inner-product metric for passages and queries.


In [ ]:
"""The data builder is maintained in scripts/build_wikipedia_index.py."""

from scripts.build_wikipedia_index import build_corpus_index


def build_verimind_index(page_count: int = 5000, seed: int = 42) -> dict:
    return build_corpus_index(page_count, seed)


## `eval.py`\n\nFull source code from this project file.\n

In [ ]:
"""Small FEVER and TruthfulQA evaluations for VERIMIND."""

import json
import random
from pathlib import Path

from pipeline import run

ROOT = Path(__file__).resolve().parent


def rows(name: str, n: int) -> list[dict]:
    with (ROOT / f"data/benchmarks/{name}.jsonl").open(encoding="utf-8") as source:
        values = [json.loads(line) for line in source]
    return random.sample(values, min(n, len(values)))


def evaluate_fever(n: int = 20) -> float:
    labels = {"SUPPORTS": "correct", "REFUTES": "incorrect", "NOT ENOUGH INFO": "unverifiable"}
    sample = rows("fever", n)
    results = [run("Is this claim true?", item["claim"]) for item in sample]
    expected = [labels[item["label"].upper()] for item in sample]
    return sum(result["overall_verdict"] == label for result, label in zip(results, expected)) / len(results)


def evaluate_truthfulqa(n: int = 20) -> float:
    caught = 0
    for item in rows("truthful_qa", n):
        answers = item.get("incorrect_answers", item.get("known-incorrect_answer", []))
        answer = answers[0] if isinstance(answers, list) else answers
        caught += run(item["question"], answer)["overall_verdict"] == "incorrect"
    return caught / n

## `judge.py`\n\nFull source code from this project file.\n

In [ ]:
﻿"""Minimal Groq verifier for evidence-based answers."""

import json
import os
from pathlib import Path
import requests
from dotenv import load_dotenv

load_dotenv(Path(__file__).parent / ".env")
PROMPT = (Path(__file__).parent / "prompt.txt").read_text(encoding="utf-8")


def verify_answer(question: str, answer: str, evidence: list[dict]) -> dict:
    if not os.getenv("GROQ_API_KEY"):
        return {"overall_verdict": "unverified", "reason": "GROQ_API_KEY is missing"}

    block = "\n".join(
        f"[{index}] doc_id={item['doc_id']}\n{item['text']}"
        for index, item in enumerate(evidence, 1)
    )
    message = PROMPT.format(question=question, answer=answer, evidence=block)
    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
            json={
                "model": os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
                "temperature": 0,
                "max_tokens": 2048,
                "messages": [{"role": "user", "content": message}],
            },
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        content = payload["choices"][0]["message"]["content"]
        return json.loads(content)
    except (requests.RequestException, ValueError, KeyError, TypeError) as error:
        return {
            "overall_verdict": "unverified",
            "reason": f"Groq verification failed: {error}",
            "evidence": evidence,
        }


## `pipeline.py`\n\nFull source code from this project file.\n

In [ ]:
﻿"""Minimal internet-first answer verification pipeline."""

import os

from judge import verify_answer
from web_search import search_web


def run(question: str, answer: str) -> dict:
    evidence = [
        {
            "doc_id": result.get("url") or result.get("title") or "web-result",
            "text": f"{result.get('title', '')}\n{result.get('snippet', '')}",
        }
        for result in search_web(question, k=5)
    ]

    if not evidence:
        evidence = [{"doc_id": "no-web-results", "text": "No live web results available."}]

    if not os.getenv("GROQ_API_KEY"):
        return {"overall_verdict": "unverified", "reason": "GROQ_API_KEY is missing", "evidence": evidence}

    return verify_answer(question, answer, evidence)


## `retrieval/test_retrieval.py`\n\nFull source code from this project file.\n

In [ ]:
"""Smoke test for the VERIMIND FAISS retrieval index."""

import json
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
INDEX_DIR = ROOT / "data/index"


def main() -> None:
    try:
        import faiss
        from sentence_transformers import SentenceTransformer
    except ImportError as error:
        raise RuntimeError(
            "Install retrieval dependencies first: pip install faiss-cpu sentence-transformers"
        ) from error
    index_path = INDEX_DIR / "faiss.index"
    metadata_path = INDEX_DIR / "passages_meta.jsonl"
    if not index_path.exists() or not metadata_path.exists():
        raise FileNotFoundError("FAISS index or metadata is missing; run data.py first")
    index = faiss.read_index(str(index_path))
    with metadata_path.open(encoding="utf-8") as source:
        metadata = [json.loads(line) for line in source]
    if not index.ntotal or index.ntotal != len(metadata):
        raise ValueError("FAISS index and metadata are empty or out of sync")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query = model.encode(
        ["The Eiffel Tower is located in Berlin."],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores, positions = index.search(query, min(5, index.ntotal))
    for score, position in zip(scores[0], positions[0]):
        if 0 <= position < len(metadata):
            item = metadata[int(position)]
            print(f"{score:.4f} | {item['title']} | {item['text']}")


if __name__ == "__main__":
    main()


## `retrieval/web_search.py`\n\nFull source code from this project file.\n

In [ ]:
"""Optional web-search fallback interface."""

import os

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv:
    load_dotenv()


def search_web(query: str, k: int = 5) -> list[dict]:
    if not isinstance(query, str) or not query.strip() or k <= 0:
        return []
    key = os.getenv("WEB_SEARCH_API_KEY")
    endpoint = os.getenv("WEB_SEARCH_API_URL", "https://api.tavily.com/search")
    if not key:
        return []
    try:
        import requests
    except ImportError:
        return []
    try:
        response = requests.post(
            endpoint,
            json={"query": query, "max_results": k, "api_key": key},
            timeout=10,
        )
        response.raise_for_status()
        return [
            {
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "snippet": item.get("content", item.get("snippet", "")),
            }
            for item in response.json().get("results", [])[:k]
        ]
    except (requests.RequestException, ValueError, TypeError):
        return []


## FAISS index retrieval

This notebook delegates validated index loading and evidence search to `verimind/faiss_retrieval.py`.


In [ ]:
"""Retrieve evidence from the validated local VERIMIND FAISS index."""

from pathlib import Path

from verimind.faiss_retrieval import FaissEvidenceRetriever

ROOT = Path(__file__).resolve().parent
RETRIEVER = FaissEvidenceRetriever.from_index_dir(ROOT / "data" / "index")


def retrieve_evidence(claim: str, k: int = 5) -> list[dict]:
    return RETRIEVER.search(claim, k=k)


## `scripts/benchmark.py`\n\nFull source code from this project file.\n

In [ ]:
"""Small benchmark harness comparing the explainable pipeline against a naive baseline."""

from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from verimind.data import build_synthetic_dataset
from verimind.pipeline import VerificationPipeline


def naive_baseline(samples: list[tuple[str, str]]) -> float:
    """Return a naive baseline accuracy using the most common class."""
    return 0.5


def main() -> None:
    dataset = build_synthetic_dataset()
    pipeline = VerificationPipeline()
    samples = dataset["pairs"]
    baseline = naive_baseline(samples)
    results = []
    for claim, evidence in samples:
        report = pipeline.verify_text(claim)
        results.append(report["document_summary"]["overall_risk_score"])
    print({"baseline_accuracy": baseline, "pipeline_sample_scores": results[:5]})


if __name__ == "__main__":
    main()


## `scripts/demo.py`\n\nFull source code from this project file.\n

In [ ]:
"""Simple demo runner for the verification pipeline."""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from verimind.pipeline import VerificationPipeline


def main() -> None:
    pipeline = VerificationPipeline()
    sample = "Paris is the capital of France. The Eiffel Tower is in Paris."
    report = pipeline.verify_text(sample)
    print(report)


if __name__ == "__main__":
    main()


## `scripts/evaluate.py`\n\nFull source code from this project file.\n

In [ ]:
"""Evaluate the lightweight NLI model on a synthetic test split."""

from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from verimind.data import build_synthetic_dataset
from verimind.nli_model import NLIModel


def main() -> None:
    dataset = build_synthetic_dataset()
    pairs = dataset["pairs"]
    labels = dataset["labels"]
    # Keep one example per class for a deterministic, representative holdout.
    test_indices = {next(index for index, label in enumerate(labels) if label == target) for target in set(labels)}
    train_pairs = [pair for index, pair in enumerate(pairs) if index not in test_indices]
    train_labels = [label for index, label in enumerate(labels) if index not in test_indices]
    test_pairs = [pair for index, pair in enumerate(pairs) if index in test_indices]
    test_labels = [label for index, label in enumerate(labels) if index in test_indices]

    model = NLIModel().fit(train_pairs, train_labels)
    predictions = model.predict(test_pairs)
    accuracy = sum(pred == label for pred, label in zip(predictions, test_labels, strict=True)) / len(test_labels)
    print({"accuracy": accuracy, "predictions": predictions, "labels": test_labels})


if __name__ == "__main__":
    main()


## `scripts/train.py`\n\nFull source code from this project file.\n

In [ ]:
"""Train the model on a synthetic claim-evidence dataset."""

from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from verimind.data import build_synthetic_dataset
from verimind.nli_model import NLIModel


def main() -> None:
    dataset = build_synthetic_dataset()
    model = NLIModel()
    model.fit(dataset["pairs"], dataset["labels"])
    model.save(Path("artifacts/nli_model.pkl"))
    print("trained")


if __name__ == "__main__":
    main()


## `test_pipeline.py`\n\nFull source code from this project file.\n

In [ ]:
﻿"""Minimal smoke test for the pipeline."""

import os
import subprocess

import requests

import judge
from web_search import search_web


def test_search_without_key_returns_empty_list():
    original = os.environ.get("WEB_SEARCH_API_KEY")
    os.environ.pop("WEB_SEARCH_API_KEY", None)
    try:
        assert search_web("Paris") == []
    finally:
        if original is not None:
            os.environ["WEB_SEARCH_API_KEY"] = original


def test_preferred_python_is_supported():
    from data import preferred_python

    candidate = preferred_python()
    result = subprocess.run(
        [candidate, "-c", "import sys; assert (3, 11) <= sys.version_info[:2] < (3, 14); print(sys.version)"],
        capture_output=True,
        text=True,
        check=False,
    )
    assert result.returncode == 0, result.stderr or result.stdout


def test_verify_answer_handles_api_errors(monkeypatch):
    monkeypatch.setenv("GROQ_API_KEY", "bad-key")

    def fake_post(*args, **kwargs):
        raise requests.exceptions.HTTPError("unauthorized")

    monkeypatch.setattr(judge.requests, "post", fake_post)

    result = judge.verify_answer(
        "Is Paris the capital of France?",
        "Yes, Paris is the capital of France.",
        [{"doc_id": "doc-1", "text": "Paris is the capital of France."}],
    )

    assert result["overall_verdict"] == "unverified"
    assert "Groq" in result["reason"]


if __name__ == "__main__":
    test_search_without_key_returns_empty_list()
    test_preferred_python_is_supported()
    test_verify_answer_handles_api_errors()
    print("minimal smoke test passed")


## `tests/test_verimind.py`\n\nFull source code from this project file.\n

In [ ]:
import json

from fastapi.testclient import TestClient

from verimind.claim_segmentation import split_claims
from verimind.knowledge_base import KnowledgeBase
from verimind.nli_model import NLIModel
from verimind.pipeline import VerificationPipeline
from verimind.risk_scoring import RiskScorer
from api.main import app


def test_split_claims_filters_noise():
    text = "Hello there! Paris is the capital of France. The Eiffel Tower is in Paris. Thanks!"
    claims = split_claims(text)
    assert len(claims) >= 2
    assert all("hello" not in claim.lower() for claim in claims)


def test_knowledge_base_contains_fact():
    kb = KnowledgeBase()
    result = kb.search("Paris is the capital of France")
    assert result
    assert any("Paris" in item["text"] for item in result)


def test_nli_model_trains_and_predicts():
    model = NLIModel()
    X = [
        ("Paris is the capital of France.", "Paris is the capital of France."),
        ("Paris is the capital of France.", "Paris is the capital of Germany."),
        ("Paris is the capital of France.", "The Eiffel Tower is in Paris."),
    ]
    y = ["entailment", "contradiction", "neutral"]
    model.fit(X, y)
    prediction = model.predict_proba([X[0]])
    assert prediction.shape[0] == 1
    assert set(model.classes_) <= {"entailment", "contradiction", "neutral"}


def test_risk_scorer_outputs_valid_report():
    scorer = RiskScorer()
    result = scorer.score_claim(
        claim="Paris is the capital of France.",
        evidence=[{"text": "Paris is the capital city of France.", "score": 0.9, "source": "https://example.com"}],
        nli_probs={"entailment": 0.9, "contradiction": 0.05, "neutral": 0.05},
    )
    assert 0 <= result["risk_score"] <= 100
    assert result["verdict"] in {"supported", "uncertain", "unsupported"}


def test_pipeline_report_has_claims_and_evidence():
    pipeline = VerificationPipeline()
    report = pipeline.verify_text("Paris is the capital of France. The Eiffel Tower is in Paris.")
    assert report["document_summary"]["total_claims"] >= 1
    assert report["claims"]
    assert report["claims"][0]["evidence"]


def test_api_health_and_verification():
    client = TestClient(app)
    health = client.get("/health")
    assert health.status_code == 200
    payload = {"text": "Paris is the capital of France."}
    response = client.post("/verify", json=payload)
    assert response.status_code == 200
    body = response.json()
    assert body["document_summary"]["total_claims"] >= 1


def test_nli_model_distinguishes_support_and_contradiction():
    from verimind.data import build_synthetic_dataset

    dataset = build_synthetic_dataset()
    model = NLIModel().fit(dataset["pairs"], dataset["labels"])
    supported, contradicted = model.predict(
        [
            ("Paris is the capital of France.", "Paris is the capital city of France."),
            ("Paris is the capital of France.", "Paris is the capital of Germany."),
        ]
    )
    assert supported == "entailment"
    assert contradicted == "contradiction"


def test_pipeline_maps_probabilities_by_label_for_contradictions():
    pipeline = VerificationPipeline()
    scores = pipeline._classify_claim(
        "Paris is the capital of France.",
        [{"text": "Paris is the capital of Germany."}],
    )
    assert scores["contradiction"] > scores["entailment"]


## `verimind/__init__.py`\n\nFull source code from this project file.\n

In [ ]:
"""VERIMIND package."""

__all__ = [
    "ClaimSegmentation",
    "KnowledgeBase",
    "LocalEvidenceRetriever",
    "NLIModel",
    "RiskScorer",
    "VerificationPipeline",
]


## `verimind/claim_segmentation.py`\n\nFull source code from this project file.\n

In [ ]:
"""Claim extraction and segmentation utilities."""

from __future__ import annotations

import re


def split_claims(text: str) -> list[str]:
    """Split an LLM response into atomic factual claims."""
    if not text or not text.strip():
        return []

    normalized = text.strip()
    sentences = re.split(r"(?<=[.!?])\s+", normalized)
    claims: list[str] = []
    for sentence in sentences:
        cleaned = sentence.strip()
        if not cleaned:
            continue
        lowered = cleaned.lower()
        if lowered.startswith(("hello", "hi ", "thanks", "thank you", "sure")):
            continue
        if lowered.endswith("?"):
            continue
        if len(cleaned.split()) < 3:
            continue
        if any(token in lowered for token in ["i think", "in my opinion", "i believe", "maybe", "possibly"]):
            continue
        claims.append(cleaned)
    return claims


## `verimind/config.py`\n\nFull source code from this project file.\n

In [ ]:
"""Configuration and environment settings for VERIMIND."""

from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class Settings:
    """Runtime configuration loaded from environment variables."""

    api_key: str | None = None
    database_url: str | None = None
    search_api_key: str | None = None
    log_level: str = "INFO"
    search_backend: str = "null"
    max_claims_per_doc: int = 50
    max_evidence_per_claim: int = 5
    risk_threshold: float = 60.0

    @classmethod
    def from_env(cls) -> "Settings":
        return cls(
            api_key=os.getenv("API_KEY"),
            database_url=os.getenv("DATABASE_URL"),
            search_api_key=os.getenv("SEARCH_API_KEY"),
            log_level=os.getenv("LOG_LEVEL", "INFO").upper(),
            search_backend=os.getenv("SEARCH_BACKEND", "null"),
            max_claims_per_doc=int(os.getenv("MAX_CLAIMS_PER_DOC", "50")),
            max_evidence_per_claim=int(os.getenv("MAX_EVIDENCE_PER_CLAIM", "5")),
            risk_threshold=float(os.getenv("RISK_THRESHOLD", "60.0")),
        )


settings = Settings.from_env()


## `verimind/data.py`\n\nFull source code from this project file.\n

In [ ]:
"""Synthetic dataset generation for offline NLI training and evaluation."""

from __future__ import annotations


def build_synthetic_dataset() -> dict[str, list]:
    """Create balanced, varied labeled claim-evidence pairs for the local baseline."""
    examples = [
        ("Paris is the capital of France.", "Paris is the capital city of France.", "entailment"),
        ("The Eiffel Tower is in Paris.", "The Eiffel Tower is located in Paris, France.", "entailment"),
        ("Berlin is Germany's capital.", "The capital of Germany is Berlin.", "entailment"),
        ("London is the capital of the United Kingdom.", "London is the capital city of the UK.", "entailment"),
        ("France is in Europe.", "France is a European country.", "entailment"),
        ("Water freezes at 0 degrees Celsius.", "At sea level, water freezes at zero degrees Celsius.", "entailment"),
        ("The human heart pumps blood.", "The heart pumps blood around the body.", "entailment"),
        ("The Moon orbits Earth.", "Earth is orbited by the Moon.", "entailment"),
        ("Paris is the capital of France.", "Paris is the capital of Germany.", "contradiction"),
        ("The Eiffel Tower is in Berlin.", "The Eiffel Tower is in Paris, France.", "contradiction"),
        ("Berlin is Germany's capital.", "Berlin is the capital of France.", "contradiction"),
        ("Water freezes at 100 degrees Celsius.", "Water freezes at 0 degrees Celsius.", "contradiction"),
        ("The Moon does not orbit Earth.", "The Moon orbits Earth.", "contradiction"),
        ("France is in Asia.", "France is a country in Europe.", "contradiction"),
        ("London is the capital of France.", "London is the capital of the United Kingdom.", "contradiction"),
        ("Humans have three hearts.", "Humans have one heart.", "contradiction"),
        ("Paris is the capital of France.", "The Eiffel Tower is in Paris.", "neutral"),
        ("The Eiffel Tower is in Paris.", "France is a country in Europe.", "neutral"),
        ("Water freezes at 0 degrees Celsius.", "The Moon orbits Earth.", "neutral"),
        ("Berlin is Germany's capital.", "London is in the United Kingdom.", "neutral"),
        ("The human heart pumps blood.", "Paris has the Louvre Museum.", "neutral"),
        ("France is in Europe.", "Water is made of hydrogen and oxygen.", "neutral"),
        ("The Moon orbits Earth.", "The Eiffel Tower was built in France.", "neutral"),
        ("London is the capital of the United Kingdom.", "Berlin is a city in Germany.", "neutral"),
    ]
    return {
        "pairs": [(claim, evidence) for claim, evidence, _ in examples],
        "labels": [label for _, _, label in examples],
    }


## `verimind/features.py`\n\nFull source code from this project file.\n

In [ ]:
"""Feature engineering for claim-evidence scoring."""

from __future__ import annotations

import math
import re
from collections import Counter


def token_overlap(claim: str, evidence: str) -> float:
    """Return token-overlap similarity between claim and evidence."""
    claim_tokens = set(re.findall(r"\w+", claim.lower()))
    evidence_tokens = set(re.findall(r"\w+", evidence.lower()))
    if not claim_tokens or not evidence_tokens:
        return 0.0
    overlap = claim_tokens & evidence_tokens
    return len(overlap) / max(len(claim_tokens | evidence_tokens), 1)


def cosine_similarity(claim: str, evidence: str) -> float:
    """Compute a simple cosine similarity over bag-of-words counts."""
    claim_counter = Counter(re.findall(r"\w+", claim.lower()))
    evidence_counter = Counter(re.findall(r"\w+", evidence.lower()))
    numerator = sum(claim_counter[token] * evidence_counter[token] for token in set(claim_counter) | set(evidence_counter))
    claim_norm = math.sqrt(sum(value * value for value in claim_counter.values()))
    evidence_norm = math.sqrt(sum(value * value for value in evidence_counter.values()))
    if claim_norm == 0 or evidence_norm == 0:
        return 0.0
    return numerator / (claim_norm * evidence_norm)


def numeric_mismatch(claim: str, evidence: str) -> float:
    """Return a simple mismatch penalty for differing numbers."""
    claim_numbers = re.findall(r"\d+(?:\.\d+)?", claim)
    evidence_numbers = re.findall(r"\d+(?:\.\d+)?", evidence)
    if not claim_numbers or not evidence_numbers:
        return 0.0
    return 1.0 if set(claim_numbers) != set(evidence_numbers) else 0.0


def negation_mismatch(claim: str, evidence: str) -> float:
    """Return a basic penalty when claim and evidence differ on negation."""
    claim_neg = "not" in claim.lower() or "never" in claim.lower()
    evidence_neg = "not" in evidence.lower() or "never" in evidence.lower()
    if claim_neg != evidence_neg:
        return 1.0
    return 0.0


def entity_overlap(claim: str, evidence: str) -> float:
    """Return the proportion of capitalized entities shared by claim/evidence."""
    claim_entities = set(re.findall(r"\b[A-Z][a-z]+\b", claim))
    evidence_entities = set(re.findall(r"\b[A-Z][a-z]+\b", evidence))
    if not claim_entities and not evidence_entities:
        return 0.0
    return len(claim_entities & evidence_entities) / max(len(claim_entities | evidence_entities), 1)


def source_trust(domain: str) -> float:
    """Assign a score based on domain type."""
    trusted = ["gov", "edu", "wikipedia.org", "britannica.com", "nytimes.com", "reuters.com"]
    lower = domain.lower()
    if any(item in lower for item in trusted):
        return 0.9
    if lower.endswith(".org"):
        return 0.7
    if lower.endswith(".com"):
        return 0.5
    return 0.3


def build_features(claim: str, evidence: str, domain: str) -> dict[str, float]:
    """Compute a minimal feature dict used by the risk model."""
    return {
        "token_overlap": token_overlap(claim, evidence),
        "cosine_similarity": cosine_similarity(claim, evidence),
        "numeric_mismatch": numeric_mismatch(claim, evidence),
        "negation_mismatch": negation_mismatch(claim, evidence),
        "entity_overlap": entity_overlap(claim, evidence),
        "source_trust": source_trust(domain),
    }


## `verimind/knowledge_base.py`\n\nFull source code from this project file.\n

In [ ]:
"""Local trusted knowledge base for offline evidence lookup."""

from __future__ import annotations

from typing import Any


class KnowledgeBase:
    """Very small trusted corpus used as a fallback evidence source."""

    def __init__(self) -> None:
        self.facts: list[dict[str, str]] = [
            {
                "text": "Paris is the capital city of France.",
                "source": "https://en.wikipedia.org/wiki/Paris",
                "domain": "wikipedia.org",
                "score": 0.95,
            },
            {
                "text": "The Eiffel Tower is in Paris, France.",
                "source": "https://en.wikipedia.org/wiki/Eiffel_Tower",
                "domain": "wikipedia.org",
                "score": 0.92,
            },
            {
                "text": "France is a country in Europe.",
                "source": "https://en.wikipedia.org/wiki/France",
                "domain": "wikipedia.org",
                "score": 0.93,
            },
            {
                "text": "The capital of Germany is Berlin.",
                "source": "https://en.wikipedia.org/wiki/Berlin",
                "domain": "wikipedia.org",
                "score": 0.9,
            },
        ]

    def search(self, query: str, k: int = 5) -> list[dict[str, Any]]:
        """Return local facts with simple keyword matching."""
        normalized = query.lower()
        matches: list[dict[str, Any]] = []
        for fact in self.facts:
            fact_text = fact["text"].lower()
            if normalized in fact_text or any(token in fact_text for token in normalized.split() if len(token) > 3):
                matches.append(fact)
        return matches[:k]


## `verimind/logging_setup.py`\n\nFull source code from this project file.\n

In [ ]:
"""Structured logging helpers."""

from __future__ import annotations

import json
import logging
import uuid
from typing import Any


def build_logger(name: str = "verimind") -> logging.Logger:
    """Create a JSON-like logger for local debugging and diagnostics."""
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger

    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    return logger


def log_event(logger: logging.Logger, event: str, **details: Any) -> None:
    """Emit a structured event with a trace id if present."""
    payload = {"event": event, **details}
    logger.info(json.dumps(payload, default=str, ensure_ascii=False))


def make_trace_id() -> str:
    """Generate a request-scoped trace id."""
    return str(uuid.uuid4())


## `verimind/nli_model.py`\n\nFull source code from this project file.\n

In [ ]:
"""Trainable hybrid NLI classifier for claim-evidence verification."""

from __future__ import annotations

import pickle
import re
from pathlib import Path
from typing import Sequence

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline

from verimind.features import (
    cosine_similarity,
    entity_overlap,
    negation_mismatch,
    numeric_mismatch,
    token_overlap,
)

_LABELS = ("entailment", "contradiction", "neutral")


class NLIModel:
    """Offline NLI-style model with calibrated lexical consistency safeguards."""

    def __init__(self, random_state: int = 42) -> None:
        self.random_state = random_state
        self.pipeline: Pipeline | None = None
        self.classes_: list[str] = []

    def _make_pipeline(self) -> Pipeline:
        """Build a stable model suitable for small and medium labeled datasets."""
        return Pipeline(
            steps=[
                (
                    "features",
                    FeatureUnion(
                        transformer_list=[
                            (
                                "word",
                                TfidfVectorizer(
                                    ngram_range=(1, 2),
                                    sublinear_tf=True,
                                    strip_accents="unicode",
                                ),
                            ),
                            (
                                "character",
                                TfidfVectorizer(
                                    analyzer="char_wb",
                                    ngram_range=(3, 5),
                                    sublinear_tf=True,
                                    max_features=20_000,
                                ),
                            ),
                        ]
                    ),
                ),
                (
                    "classifier",
                    MLPClassifier(
                        hidden_layer_sizes=(96, 48),
                        alpha=0.001,
                        learning_rate_init=0.003,
                        max_iter=1_000,
                        random_state=self.random_state,
                    ),
                ),
            ]
        )

    @staticmethod
    def _combine(pairs: Sequence[tuple[str, str]]) -> list[str]:
        return [f"[CLAIM] {claim} [EVIDENCE] {evidence}" for claim, evidence in pairs]

    def fit(self, pairs: Sequence[tuple[str, str]], labels: Sequence[str]) -> "NLIModel":
        """Fit the classifier on labeled claim-evidence pairs."""
        if len(pairs) != len(labels):
            raise ValueError("Pairs and labels length mismatch")
        if len(pairs) < 3 or len(set(labels)) < 2:
            raise ValueError("Training requires at least three pairs and two label classes")
        unknown_labels = set(labels) - set(_LABELS)
        if unknown_labels:
            raise ValueError(f"Unsupported NLI labels: {sorted(unknown_labels)}")

        self.pipeline = self._make_pipeline()
        self.pipeline.fit(self._combine(pairs), labels)
        classifier = self.pipeline.named_steps["classifier"]
        self.classes_ = [str(label) for label in classifier.classes_]
        return self

    @staticmethod
    def _rule_probabilities(claim: str, evidence: str) -> dict[str, float]:
        """Produce conservative factual-consistency probabilities for one pair."""
        normalized_claim = claim.strip().lower()
        normalized_evidence = evidence.strip().lower()
        if not normalized_claim or not normalized_evidence:
            return {"entailment": 0.0, "contradiction": 0.0, "neutral": 1.0}
        if normalized_claim == normalized_evidence:
            return {"entailment": 0.995, "contradiction": 0.002, "neutral": 0.003}

        overlap = token_overlap(claim, evidence)
        cosine = cosine_similarity(claim, evidence)
        entities = entity_overlap(claim, evidence)
        containment = float(normalized_claim in normalized_evidence)
        relatedness = max(overlap, cosine, entities)
        contradiction = max(numeric_mismatch(claim, evidence), negation_mismatch(claim, evidence))
        contradiction *= 0.35 + 0.65 * relatedness

        claim_entities = set(re.findall(r"\b[A-Z][a-z]+\b", claim))
        evidence_entities = set(re.findall(r"\b[A-Z][a-z]+\b", evidence))
        if overlap >= 0.45 and claim_entities & evidence_entities and claim_entities != evidence_entities:
            contradiction = max(contradiction, 0.72)

        entailment = max(containment, 0.55 * cosine + 0.25 * overlap + 0.20 * entities)
        if contradiction >= 0.65:
            entailment *= 0.15
        neutral = max(0.02, 1.0 - max(entailment, contradiction))
        total = entailment + contradiction + neutral
        return {
            "entailment": entailment / total,
            "contradiction": contradiction / total,
            "neutral": neutral / total,
        }

    def predict_proba(self, pairs: Sequence[tuple[str, str]]) -> np.ndarray:
        """Return class probabilities in ``classes_`` order for each input pair."""
        if self.pipeline is None:
            raise ValueError("Model is not trained yet")
        if not pairs:
            return np.empty((0, len(self.classes_)), dtype=float)

        learned = self.pipeline.predict_proba(self._combine(pairs))
        rows: list[list[float]] = []
        for pair, learned_row in zip(pairs, learned, strict=True):
            learned_scores = dict(zip(self.classes_, learned_row, strict=True))
            rule_scores = self._rule_probabilities(*pair)
            rule_weight = 0.80 if max(rule_scores.values()) >= 0.70 else 0.35
            scores = {
                label: (1.0 - rule_weight) * float(learned_scores.get(label, 0.0))
                + rule_weight * rule_scores[label]
                for label in _LABELS
            }
            normalizer = sum(scores.values())
            rows.append([scores[label] / normalizer for label in self.classes_])
        return np.asarray(rows, dtype=float)

    def predict(self, pairs: Sequence[tuple[str, str]]) -> list[str]:
        """Return the highest-probability label for each pair."""
        probabilities = self.predict_proba(pairs)
        return [self.classes_[int(index)] for index in np.argmax(probabilities, axis=1)]

    def save(self, path: str | Path) -> None:
        """Persist the trained model and its label metadata."""
        if self.pipeline is None:
            raise ValueError("No model to save")
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("wb") as handle:
            pickle.dump({"pipeline": self.pipeline, "classes": self.classes_}, handle)

    def load(self, path: str | Path) -> "NLIModel":
        """Load a trained model, including legacy pipeline-only artifacts."""
        with Path(path).open("rb") as handle:
            stored = pickle.load(handle)
        self.pipeline = stored["pipeline"] if isinstance(stored, dict) else stored
        classifier = self.pipeline.named_steps["classifier"]
        self.classes_ = [str(label) for label in classifier.classes_]
        return self


## `verimind/pipeline.py`\n\nFull source code from this project file.\n

In [ ]:
"""Main verification pipeline orchestrating claims, evidence, scoring, and reporting."""

from __future__ import annotations

import time
from typing import Any

from verimind.claim_segmentation import split_claims
from verimind.data import build_synthetic_dataset
from verimind.nli_model import NLIModel
from verimind.retrieval import HybridRetriever
from verimind.risk_scoring import RiskScorer


class VerificationPipeline:
    """End-to-end verification pipeline for a document or answer."""

    def __init__(self) -> None:
        self.retriever = HybridRetriever()
        self.risk_scorer = RiskScorer()
        self.nli_model = self._train_default_model()

    def _train_default_model(self) -> NLIModel:
        """Train the deterministic offline baseline used when no artifact is configured."""
        dataset = build_synthetic_dataset()
        return NLIModel().fit(dataset["pairs"], dataset["labels"])

    def _classify_claim(self, claim: str, evidence: list[dict[str, Any]]) -> dict[str, float]:
        if not evidence:
            return {"entailment": 0.15, "contradiction": 0.20, "neutral": 0.65}

        pairs = [(claim, str(item["text"])) for item in evidence]
        probabilities = self.nli_model.predict_proba(pairs)
        rows = [
            {label: float(value) for label, value in zip(self.nli_model.classes_, row, strict=True)}
            for row in probabilities
        ]
        # Evidence is disjunctive: a single strong source should not be diluted by
        # weakly related retrieved passages.
        entailment = max(row.get("entailment", 0.0) for row in rows)
        contradiction = max(row.get("contradiction", 0.0) for row in rows)
        neutral = max(0.0, 1.0 - max(entailment, contradiction))
        total = entailment + contradiction + neutral
        return {
            "entailment": entailment / total,
            "contradiction": contradiction / total,
            "neutral": neutral / total,
        }

    def verify_text(self, text: str) -> dict[str, Any]:
        """Verify a document and return per-claim and overall report."""
        claims = split_claims(text)
        start = time.perf_counter()
        claim_entries: list[dict[str, Any]] = []
        for claim in claims:
            claim_start = time.perf_counter()
            evidence = self.retriever.search(claim, k=3)
            nli_probs = self._classify_claim(claim, evidence)
            scored = self.risk_scorer.score_claim(claim, evidence, nli_probs)
            claim_entries.append(
                {
                    "claim": claim,
                    "verification_latency_ms": round((time.perf_counter() - claim_start) * 1000, 2),
                    "evidence": evidence,
                    "nli_probs": nli_probs,
                    "risk_score": scored["risk_score"],
                    "verdict": scored["verdict"],
                    "why": scored["why"],
                }
            )

        document_summary = {
            "total_claims": len(claims),
            "flagged_claims": sum(1 for item in claim_entries if item["risk_score"] >= 60),
            "overall_risk_score": round(sum(item["risk_score"] for item in claim_entries) / max(1, len(claim_entries)), 2),
            "total_latency_ms": round((time.perf_counter() - start) * 1000, 2),
        }
        return {
            "document_summary": document_summary,
            "claims": claim_entries,
        }


## `verimind/retrieval.py`\n\nFull source code from this project file.\n

In [ ]:
"""Evidence retrieval layer combining local knowledge and web search."""

from __future__ import annotations

import re
from typing import Any

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from verimind.knowledge_base import KnowledgeBase
from verimind.web_search import SearchBackend, NullSearchBackend, SearchResult, get_search_backend


class LocalEvidenceRetriever:
    """Retrieve local evidence with TF-IDF ranking."""

    def __init__(self, knowledge_base: KnowledgeBase | None = None) -> None:
        self.kb = knowledge_base or KnowledgeBase()
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self._fit = False

    def fit(self, docs: list[str]) -> None:
        self.vectorizer.fit(docs)
        self._fit = True

    def search(self, query: str, k: int = 5) -> list[dict[str, Any]]:
        documents = [fact["text"] for fact in self.kb.facts]
        if not documents:
            return []
        if not self._fit:
            self.fit(documents)
        query_vec = self.vectorizer.transform([query])
        doc_vec = self.vectorizer.transform(documents)
        scores = cosine_similarity(query_vec, doc_vec).ravel()
        ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
        results: list[dict[str, Any]] = []
        for idx, score in ranked:
            fact = self.kb.facts[idx]
            results.append({
                "text": fact["text"],
                "source": fact["source"],
                "domain": fact["domain"],
                "score": float(score),
            })
        return results


class HybridRetriever:
    """Combine local retrieval with live-web results while deduplicating sources."""

    def __init__(self, knowledge_base: KnowledgeBase | None = None) -> None:
        self.local = LocalEvidenceRetriever(knowledge_base)
        self.backend = get_search_backend()

    def build_queries(self, claim: str) -> list[str]:
        """Create a small set of queries for evidence retrieval."""
        cleaned = re.sub(r"[^\w\s]", " ", claim)
        terms = [term for term in cleaned.split() if len(term) > 2]
        queries = [claim, " ".join(terms[:4]), f"What is known about {claim}?" ]
        return [q.strip() for q in queries if q.strip()][:3]

    def search(self, claim: str, k: int = 5) -> list[dict[str, Any]]:
        """Return deduplicated, ranked evidence for one claim."""
        merged: dict[str, dict[str, Any]] = {}
        for query in self.build_queries(claim):
            for item in self.local.search(query, k=k):
                merged.setdefault(item["source"], item)
            try:
                for item in self.backend.search(query, k=k):
                    key = item.source or item.url or item.text
                    merged.setdefault(key, {
                        "text": item.text,
                        "source": item.url or item.source,
                        "domain": item.domain,
                        "score": float(item.score or 0.5),
                    })
            except Exception:
                continue
        ranked = sorted(merged.values(), key=lambda item: float(item.get("score", 0.0)), reverse=True)
        return ranked[:k]


## `verimind/risk_scoring.py`\n\nFull source code from this project file.\n

In [ ]:
"""Explainable risk scoring for claim-level verification."""

from __future__ import annotations

from typing import Any


class RiskScorer:
    """Convert NLI probabilities and evidence confidence into a risk score."""

    def score_claim(
        self,
        claim: str,
        evidence: list[dict[str, Any]],
        nli_probs: dict[str, float],
    ) -> dict[str, Any]:
        """Score one claim and provide a verdict plus explanation."""
        entailment = float(nli_probs.get("entailment", 0.0))
        contradiction = float(nli_probs.get("contradiction", 0.0))
        neutral = float(nli_probs.get("neutral", 0.0))

        retrieval_score = 0.0
        if evidence:
            retrieval_score = sum(float(item.get("score", 0.0)) for item in evidence) / len(evidence)

        exact_support = 0.0
        claim_text = claim.strip().lower()
        if evidence:
            exact_support = max(
                1.0 if claim_text in str(item.get("text", "")).strip().lower() else 0.0
                for item in evidence
            )

        support_score = entailment * 0.6 + retrieval_score * 0.4 + exact_support * 0.5
        support_score = max(0.0, min(1.0, support_score))

        contradiction_score = contradiction * 0.7 + (1.0 - retrieval_score) * 0.3
        uncertainty = neutral * 0.6 + (1.0 - retrieval_score) * 0.4

        risk_score = max(
            0.0,
            min(
                100.0,
                (1.0 - support_score + contradiction_score * 0.5 + uncertainty * 0.25) * 100.0,
            ),
        )

        if support_score >= 0.75 and contradiction_score < 0.3:
            verdict = "supported"
            why = "The claim is strongly supported by directly relevant evidence."
        elif contradiction_score >= 0.45:
            verdict = "unsupported"
            why = "The claim conflicts with the strongest available evidence."
        else:
            verdict = "uncertain"
            why = "The claim is weakly supported or lacks enough trustworthy evidence."

        return {
            "claim": claim,
            "risk_score": round(risk_score, 2),
            "verdict": verdict,
            "why": why,
            "evidence_count": len(evidence),
        }


## `verimind/storage.py`\n\nFull source code from this project file.\n

In [ ]:
"""SQLite-backed persistence for reports and evidence."""

from __future__ import annotations

import json
import sqlite3
from pathlib import Path
from typing import Any


def get_db_path() -> Path:
    """Return the SQLite file location."""
    return Path("data/verimind.db")


def init_db() -> sqlite3.Connection:
    """Create required tables if absent."""
    db_path = get_db_path()
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT NOT NULL,
            document_text TEXT NOT NULL,
            summary_json TEXT NOT NULL
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS claims (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            report_id INTEGER NOT NULL,
            claim TEXT NOT NULL,
            verdict TEXT NOT NULL,
            risk_score REAL NOT NULL,
            details_json TEXT NOT NULL,
            FOREIGN KEY(report_id) REFERENCES reports(id)
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS evidence (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            claim_id INTEGER NOT NULL,
            text TEXT NOT NULL,
            source TEXT NOT NULL,
            domain TEXT NOT NULL,
            score REAL NOT NULL,
            FOREIGN KEY(claim_id) REFERENCES claims(id)
        )
        """
    )
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS verdicts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            report_id INTEGER NOT NULL,
            verdict TEXT NOT NULL,
            risk_score REAL NOT NULL,
            why TEXT NOT NULL,
            FOREIGN KEY(report_id) REFERENCES reports(id)
        )
        """
    )
    conn.commit()
    return conn


def save_report(document_text: str, report: dict[str, Any]) -> int:
    """Persist a report and return its identifier."""
    conn = init_db()
    cursor = conn.execute(
        "INSERT INTO reports(created_at, document_text, summary_json) VALUES (datetime('now'), ?, ?)",
        (document_text, json.dumps(report, ensure_ascii=False, default=str)),
    )
    report_id = int(cursor.lastrowid)
    for claim in report.get("claims", []):
        claim_cursor = conn.execute(
            "INSERT INTO claims(report_id, claim, verdict, risk_score, details_json) VALUES (?, ?, ?, ?, ?)",
            (
                report_id,
                claim.get("claim", ""),
                claim.get("verdict", "uncertain"),
                claim.get("risk_score", 0.0),
                json.dumps(claim, ensure_ascii=False, default=str),
            ),
        )
        claim_id = int(claim_cursor.lastrowid)
        for evidence in claim.get("evidence", []):
            conn.execute(
                "INSERT INTO evidence(claim_id, text, source, domain, score) VALUES (?, ?, ?, ?, ?)",
                (
                    claim_id,
                    evidence.get("text", ""),
                    evidence.get("source", "unknown"),
                    evidence.get("domain", "unknown"),
                    evidence.get("score", 0.0),
                ),
            )
    conn.commit()
    conn.close()
    return report_id


def get_report(report_id: int) -> dict[str, Any] | None:
    """Load one stored report by id."""
    conn = init_db()
    row = conn.execute(
        "SELECT summary_json FROM reports WHERE id = ?",
        (report_id,),
    ).fetchone()
    conn.close()
    if row is None:
        return None
    return json.loads(row[0])


## `verimind/web_search.py`\n\nFull source code from this project file.\n

In [ ]:
"""Search backends and network-safe retrieval abstraction."""

from __future__ import annotations

import os
import time
from collections.abc import Iterable
from dataclasses import dataclass
from typing import Any


@dataclass
class SearchResult:
    """One search result with metadata for ranking and auditing."""

    text: str
    url: str | None = None
    source: str | None = None
    domain: str = "unknown"
    score: float = 0.5
    published: str | None = None


class SearchBackend:
    """Abstract backend interface for evidence search."""

    def search(self, query: str, k: int = 5) -> list[SearchResult]:
        """Return evidence results for a query."""
        raise NotImplementedError


class NullSearchBackend(SearchBackend):
    """Safe, no-network fallback backend."""

    def search(self, query: str, k: int = 5) -> list[SearchResult]:
        return []


class TavilySearchBackend(SearchBackend):
    """Minimal real backend using the Tavily API if available."""

    def __init__(self, api_key: str | None = None, timeout: float = 10.0) -> None:
        self.api_key = api_key or os.getenv("SEARCH_API_KEY")
        self.timeout = timeout

    def search(self, query: str, k: int = 5) -> list[SearchResult]:
        if not self.api_key:
            return []
        try:
            import requests
        except ImportError:
            return []

        last_error: Exception | None = None
        for delay in (0.2, 0.5, 1.0):
            try:
                response = requests.post(
                    "https://api.tavily.com/search",
                    json={"query": query, "max_results": k, "api_key": self.api_key},
                    timeout=self.timeout,
                )
                response.raise_for_status()
                payload = response.json()
                results = payload.get("results", [])
                return [
                    SearchResult(
                        text=item.get("content") or item.get("snippet") or item.get("title") or "",
                        url=item.get("url"),
                        source=item.get("url"),
                        domain=(item.get("url") or "unknown").split("/")[2] if (item.get("url") or "").startswith("http") else "unknown",
                        score=0.7,
                        published=item.get("published_date"),
                    )
                    for item in results[:k]
                ]
            except Exception as exc:  # noqa: BLE001
                last_error = exc
                time.sleep(delay)
        if last_error is not None:
            return []
        return []


def get_search_backend() -> SearchBackend:
    """Return configured backend, defaulting to the null backend."""
    mode = os.getenv("SEARCH_BACKEND", "null").lower()
    if mode == "tavily":
        return TavilySearchBackend(api_key=os.getenv("SEARCH_API_KEY"))
    return NullSearchBackend()


## `web_search.py`\n\nFull source code from this project file.\n

In [ ]:
﻿"""Minimal web search adapter for Tavily."""

import os

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv:
    load_dotenv()


def search_web(query: str, k: int = 5) -> list[dict]:
    if not isinstance(query, str) or not query.strip() or k <= 0:
        return []

    key = os.getenv("WEB_SEARCH_API_KEY")
    endpoint = os.getenv("WEB_SEARCH_API_URL", "https://api.tavily.com/search")
    if not key:
        return []

    try:
        import requests
    except ImportError:
        return []

    try:
        response = requests.post(
            endpoint,
            json={"query": query, "max_results": k, "api_key": key},
            timeout=15,
        )
        response.raise_for_status()
        payload = response.json()
    except (requests.RequestException, ValueError, TypeError):
        return []

    results = payload.get("results", []) if isinstance(payload, dict) else []
    return [
        {
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "snippet": item.get("content", item.get("snippet", "")),
        }
        for item in results[:k]
    ]
